# ActionShap — Paper Asset Generator

This reporting notebook converts completed `run_recommendation.py` JSON outputs into publication-ready tables, figures, CSV data products, and a reproducibility manifest. It **does not train models or recompute Shapley values**.

## Output

Assets are written to `code/results/paper_assets/`:

- `figures/`: PNG and PDF plots;
- `tables/`: CSV and LaTeX tables;
- `data/`: flattened seed/user metrics;
- `manifests/`: validation and provenance JSON;
- `README.md`: generated-asset notes.

The primary scientific comparison is **joint intervention budget B=2**. B=1 leave-one-out is a diagnostic oracle only.


## Prerequisites

Run the experiment for at least five seeds and place files such as these in `code/results/raw/`:

```text
movielens_actionshap_seed42.json
movielens_actionshap_seed43.json
...
```

Run this notebook after the experiment. Missing files, missing methods, and missing convergence outputs are reported; no values are fabricated.


In [ ]:
from __future__ import annotations
import json, platform, re, sys
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the repository independent of the Jupyter working directory.
candidates = [Path.cwd() / "paper-ideas/ActionShap", Path.cwd() / "..", Path.cwd()]
ACTIONSHAP_ROOT = next((p.resolve() for p in candidates if (p / "code").is_dir()), None)
if ACTIONSHAP_ROOT is None:
    raise RuntimeError("Cannot locate paper-ideas/ActionShap; set ACTIONSHAP_ROOT manually.")
CODE_ROOT = ACTIONSHAP_ROOT / "code"
RAW_ROOT = CODE_ROOT / "results" / "raw"
ASSET_ROOT = CODE_ROOT / "results" / "paper_assets"
FIG_ROOT, TABLE_ROOT = ASSET_ROOT / "figures", ASSET_ROOT / "tables"
DATA_ROOT, MANIFEST_ROOT = ASSET_ROOT / "data", ASSET_ROOT / "manifests"
for p in (FIG_ROOT, TABLE_ROOT, DATA_ROOT, MANIFEST_ROOT): p.mkdir(parents=True, exist_ok=True)
SEED_PATTERN = re.compile(r"seed(?P<seed>\d+)")
METHOD_ORDER = ["shapley_mc", "lime", "loo_oracle"]
METHOD_LABELS = {"shapley_mc": "Monte Carlo Shapley", "lime": "LIME", "loo_oracle": "LOO oracle"}
plt.rcParams.update({"figure.dpi": 140, "savefig.dpi": 300, "font.size": 10})
print(ACTIONSHAP_ROOT)


In [ ]:
def load_json_results():
    files = sorted(RAW_ROOT.glob("movielens_actionshap_seed*.json")) or sorted(RAW_ROOT.glob("*.json"))
    results, errors = [], []
    for path in files:
        try:
            obj = json.loads(path.read_text())
            match = SEED_PATTERN.search(path.name)
            obj["_file"] = str(path)
            obj["_seed"] = int(match.group("seed")) if match else obj.get("config", {}).get("seed")
            results.append(obj)
        except Exception as exc:
            errors.append({"file": str(path), "error": repr(exc)})
    return files, results, errors

FILES, RESULTS, LOAD_ERRORS = load_json_results()
print(f"Found {len(RESULTS)} valid result files.")
if LOAD_ERRORS: display(pd.DataFrame(LOAD_ERRORS))


In [ ]:
# Strict schema validation before any aggregation.
REQUIRED = {"config", "dataset", "metrics", "users"}
REQUIRED_METRICS = {"aia", "aia_null_mean", "joint_effect_b2", "joint_regret_b2_on_oracle_subset"}
validation = {"files": [str(p) for p in FILES], "errors": [], "warnings": [], "seeds": sorted({r.get("_seed") for r in RESULTS})}
if len(RESULTS) < 5: validation["warnings"].append("Fewer than five seeds were found.")
for r in RESULTS:
    name = Path(r["_file"]).name
    missing = REQUIRED - set(r)
    if missing: validation["errors"].append({"file": name, "missing": sorted(missing)}); continue
    missing = REQUIRED_METRICS - set(r["metrics"])
    if missing: validation["errors"].append({"file": name, "missing_metrics": sorted(missing)})
    if r.get("_seed") is None: validation["warnings"].append(f"{name}: seed not found")
if validation["errors"]: raise ValueError(validation["errors"])
validation["status"] = "PASS"
(MANIFEST_ROOT / "validation_report.json").write_text(json.dumps(validation, indent=2))
display(pd.DataFrame([validation]))


In [ ]:
def metric_rows(metric):
    rows = []
    for r in RESULTS:
        for method, summary in r["metrics"].get(metric, {}).items():
            if isinstance(summary, dict):
                rows.append({"seed": r.get("_seed"), "method": method, "method_label": METHOD_LABELS.get(method, method), "mean": summary.get("mean"), "median": summary.get("median"), "n": summary.get("n"), "source_file": Path(r["_file"]).name})
    return pd.DataFrame(rows)

def flatten_users():
    rows = []
    for r in RESULTS:
        for u in r.get("users", []):
            for method, aia in u.get("aia", {}).items():
                joint = u.get("joint", {}).get(method, {})
                rows.append({"seed": r.get("_seed"), "user": u.get("user"), "n_players": u.get("n_players"), "method": method, "method_label": METHOD_LABELS.get(method, method), "aia": aia, "joint_effect_b2": joint.get("effect"), "joint_regret_b2": joint.get("regret"), "efficiency_error": u.get("efficiency_error")})
    return pd.DataFrame(rows)

seed_aia = metric_rows("aia")
seed_null = metric_rows("aia_null_mean")
seed_effect = metric_rows("joint_effect_b2")
seed_regret = metric_rows("joint_regret_b2_on_oracle_subset")
user_metrics = flatten_users()
seed_metrics = pd.concat([seed_aia.assign(metric="aia"), seed_null.assign(metric="aia_null_mean"), seed_effect.assign(metric="joint_effect_b2"), seed_regret.assign(metric="joint_regret_b2")], ignore_index=True)
seed_metrics.to_csv(DATA_ROOT / "seed_metrics.csv", index=False)
user_metrics.to_csv(DATA_ROOT / "user_metrics.csv", index=False)


In [ ]:
def ci(series):
    x = pd.to_numeric(series, errors="coerce").dropna().to_numpy(float)
    if not len(x): return (np.nan, np.nan, np.nan, 0)
    mean = x.mean()
    if len(x) == 1: return (mean, mean, mean, 1)
    half = 1.96 * x.std(ddof=1) / np.sqrt(len(x))
    return (mean, mean-half, mean+half, len(x))

def summarize(frame, label):
    rows = []
    for method, group in frame.groupby("method", sort=False):
        mean, low, high, n = ci(group["mean"])
        rows.append({"metric": label, "method": method, "method_label": METHOD_LABELS.get(method, method), "mean": mean, "ci95_low": low, "ci95_high": high, "n_seeds": n})
    return pd.DataFrame(rows)

SUMMARY = {
    "aia": summarize(seed_aia, "AIA"),
    "null": summarize(seed_null, "AIA permutation null"),
    "effect": summarize(seed_effect, "Joint effect B=2"),
    "regret": summarize(seed_regret, "Joint regret B=2"),
}
for key, frame in SUMMARY.items(): frame.to_csv(TABLE_ROOT / f"{key}.csv", index=False)
display(SUMMARY["aia"])


In [ ]:
def ordered(frame):
    rank = {m: i for i, m in enumerate(METHOD_ORDER)}
    return frame.assign(_rank=frame.method.map(rank).fillna(999)).sort_values("_rank").drop(columns="_rank")

def write_tex(frame, filename, caption, label):
    tex = ordered(frame).to_latex(index=False, escape=True, float_format=lambda x: f"{x:.4f}")
    (TABLE_ROOT / filename).write_text(f"% Generated by 01_generate_paper_assets.ipynb\n\begin{{table}}[t]\centering\small\n\caption{{{caption}}}\label{{{label}}}\n{tex}\end{{table}}\n")

write_tex(SUMMARY["aia"], "aia.tex", "Attribution--Intervention Alignment across seeds.", "tab:aia")
write_tex(SUMMARY["null"], "aia_null.tex", "Within-user permutation null for AIA.", "tab:aia-null")
write_tex(SUMMARY["effect"], "joint_effect.tex", "Realized effect of selected B=2 joint interventions.", "tab:joint-effect")
write_tex(SUMMARY["regret"], "joint_regret.tex", "B=2 intervention regret.", "tab:joint-regret")

first = RESULTS[0]
protocol = pd.DataFrame([{"field": k, "value": v} for k, v in {
    "dataset": "MovieLens-1M", "evaluated_users": first["dataset"].get("evaluated_users"), "items": first["dataset"].get("items"), "candidate_recall": first["dataset"].get("candidate_recall"), "n_max": first["config"].get("n_max"), "candidate_k": first["config"].get("candidate_k"), "MC_permutations": first["config"].get("permutations"), "primary_budget": "B=2", "B=1": "diagnostic LOO oracle"}.items()])
protocol.to_csv(TABLE_ROOT / "protocol.csv", index=False)
write_tex(protocol, "protocol.tex", "Dataset and experiment protocol.", "tab:protocol")


## Figures

Each figure is saved as both PDF and 300-DPI PNG. Error bars are 95% normal-approximation confidence intervals over seeds. The CSV files remain the canonical numerical assets.


In [ ]:
def savefig(name):
    plt.tight_layout(); plt.savefig(FIG_ROOT / f"{name}.png", dpi=300, bbox_inches="tight"); plt.savefig(FIG_ROOT / f"{name}.pdf", bbox_inches="tight"); plt.show()

def error_plot(summary, ylabel, title, filename, color):
    p = ordered(summary); x = np.arange(len(p)); fig, ax = plt.subplots(figsize=(6.2, 3.8))
    ax.errorbar(x, p["mean"], yerr=[p["mean"]-p["ci95_low"], p["ci95_high"]-p["mean"]], fmt="o", capsize=4, color=color)
    ax.set_xticks(x, p["method_label"], rotation=20, ha="right"); ax.set_ylabel(ylabel); ax.set_title(title); ax.axhline(0, color="black", lw=.7); savefig(filename)

recall = pd.DataFrame([{"seed": r.get("_seed"), "recall": r["dataset"].get("candidate_recall")} for r in RESULTS])
fig, ax = plt.subplots(figsize=(5.4, 3.2)); ax.bar(recall["seed"].astype(str), recall["recall"], color="#4C78A8"); ax.set_ylim(0, 1.05); ax.set_xlabel("Seed"); ax.set_ylabel("Candidate recall"); ax.set_title("Fixed-candidate retrieval quality"); savefig("fig01_candidate_recall")
error_plot(SUMMARY["aia"], "Spearman AIA", "Attribution–Intervention Alignment", "fig02_aia_by_method", "#2F4B7C")
error_plot(SUMMARY["effect"], "Change in NDCG@K", "Selected joint intervention effect (B=2)", "fig04_joint_effect_b2", "#59A14F")
error_plot(SUMMARY["regret"], "Intervention regret", "Joint intervention regret (B=2)", "fig05_joint_regret_b2", "#E15759")


In [ ]:
# Observed AIA versus the within-user permutation null.
obs, null = SUMMARY["aia"].set_index("method"), SUMMARY["null"].set_index("method")
methods = [m for m in METHOD_ORDER if m in obs.index and m in null.index]
x = np.arange(len(methods)); width = .36; fig, ax = plt.subplots(figsize=(6.2, 3.8))
ax.bar(x-width/2, obs.loc[methods, "mean"], width, label="Observed", color="#2F4B7C")
ax.bar(x+width/2, null.loc[methods, "mean"], width, label="Permutation null", color="#BDBDBD")
ax.set_xticks(x, [METHOD_LABELS[m] for m in methods], rotation=20, ha="right"); ax.set_ylabel("AIA"); ax.set_title("Observed alignment versus chance"); ax.legend(frameon=False); savefig("fig03_aia_vs_null")


## Optional convergence figure

A convergence figure is generated only when `code/results/raw/convergence_*.json` exists with the fields `permutations`, `mean_rank_correlation_to_reference`, and `std_rank_correlation_to_reference`. Otherwise the manifest records it as pending.


In [ ]:
convergence_files = sorted(RAW_ROOT.glob("convergence_*.json"))
if convergence_files:
    rows = []
    for path in convergence_files:
        payload = json.loads(path.read_text()); rows.extend(payload if isinstance(payload, list) else payload.get("rows", []))
    convergence = pd.DataFrame(rows); convergence.to_csv(DATA_ROOT / "convergence.csv", index=False)
    fig, ax = plt.subplots(figsize=(6.2, 3.8)); ax.errorbar(convergence["permutations"], convergence["mean_rank_correlation_to_reference"], yerr=convergence["std_rank_correlation_to_reference"], marker="o", capsize=3, color="#F28E2B"); ax.set_xlabel("Monte Carlo permutations"); ax.set_ylabel("Rank correlation with reference"); ax.set_title("Monte Carlo convergence"); savefig("fig06_mc_convergence")
else:
    print("No convergence JSON found; convergence figure remains pending.")


In [ ]:
# Provenance manifest and generated README.
figures = sorted(p.name for p in FIG_ROOT.iterdir() if p.suffix in {".png", ".pdf"})
tables = sorted(p.name for p in TABLE_ROOT.iterdir() if p.suffix in {".csv", ".tex"})
manifest = {"generated_at_utc": datetime.now(timezone.utc).isoformat(), "python": sys.version, "platform": platform.platform(), "source_files": [str(p) for p in FILES], "seeds": validation["seeds"], "validation": validation, "figures": figures, "tables": tables, "notes": ["B=1 is a leave-one-out diagnostic oracle.", "B=2 is the primary joint intervention comparison.", "Prefix-walk efficiency is telescoping and not a convergence certificate.", "AIA uses a within-user permutation null."]}
(MANIFEST_ROOT / "asset_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
(ASSET_ROOT / "README.md").write_text(f"Generated by 01_generate_paper_assets.ipynb.\n\nSource files: {len(FILES)}\nSeeds: {validation['seeds']}\nValidation: {validation['status']}\nPrimary budget: B=2\nWarnings: {len(validation['warnings'])}\n")
print(json.dumps({"figures": figures, "tables": tables}, indent=2))


## Final checklist

- [ ] At least five seeds are present.
- [ ] Candidate recall is reported.
- [ ] Validation has no schema errors.
- [ ] AIA null is near zero.
- [ ] B=1 is labelled as a diagnostic oracle.
- [ ] B=2 is the headline comparison.
- [ ] Regret is reported only for valid oracle users.
- [ ] Convergence is either generated or explicitly marked pending.
- [ ] The manifest lists every generated asset.
- [ ] No result claim is written before inspecting the confidence intervals.
